# vibetrack — Jupyter Viewer

This notebook demonstrates the `--viewer=jupyter` backend, which embeds the full vibetrack web dashboard as a live iframe directly inside your notebook.

**No browser tab switching.** The dashboard lives next to your code, updates automatically as you log data, and survives cell re-runs without losing the server.

---

**Contents**
1. [Quick start](#1.-Quick-start)
2. [Train while watching live](#2.-Train-while-watching-live)
3. [Refresh the iframe in-place](#3.-Refresh-the-iframe-in-place)
4. [Compare multiple experiments](#4.-Compare-multiple-experiments)
5. [Log hyperparameters](#5.-Log-hyperparameters)
6. [Filter to specific experiments](#6.-Filter-to-specific-experiments)
7. [Get the server URL](#7.-Get-the-server-URL)
8. [JupyterHub / remote environments](#8.-JupyterHub-/-remote-environments)

In [ ]:
%pip install vibetrack -q

## 1. Quick start

`JupyterOutput` starts the web server on a background thread (picks a free port automatically when `port=0`), then renders an iframe in the cell output.  
The kernel is **never blocked** — you can keep running cells below.

In [ ]:
from vibetrack.viewers.jupyter import JupyterOutput

viewer = JupyterOutput()
viewer.show(height=700)

> **Tip:** The server reuses the same process on every `show()` call — calling this cell again just updates the iframe, it does not spawn a second server.

---

## 2. Train while watching live

Run the training loop below and watch the dashboard update in real time.  
The web UI polls `/api/data` every few seconds — no extra code needed.

In [ ]:
import math
import random
import time
from vibetrack import SummaryWriter

random.seed(42)

LR = 0.05
STEPS = 150

writer = SummaryWriter("runs/sgd")

for step in range(STEPS):
    # Synthetic training curves
    decay = math.exp(-step * LR * 0.4)
    train_loss = decay * 2.0 + 0.08 + random.gauss(0, 0.03)
    val_loss   = decay * 2.1 + 0.12 + random.gauss(0, 0.05)
    accuracy   = max(0.0, min(1.0, 1.0 - train_loss * 0.38 + random.gauss(0, 0.01)))
    grad_norm  = decay * 3.5 + random.gauss(0, 0.1)

    writer.add_scalar("loss/train", train_loss, step)
    writer.add_scalar("loss/val",   val_loss,   step)
    writer.add_scalar("accuracy",   accuracy,   step)
    writer.add_scalar("grad_norm",  grad_norm,  step)

    if step % 25 == 0:
        writer.add_text(
            "status",
            f"step={step}  loss={train_loss:.4f}  acc={accuracy:.3f}",
            step,
        )
        # Weight distribution (simulated)
        weights = [random.gauss(0, 1.0 / (1 + step * 0.02)) for _ in range(500)]
        writer.add_histogram("weights/layer1", weights, step)

    time.sleep(0.05)  # slow down so you can watch the live updates

writer.close()
print(f"Done. Final train loss: {train_loss:.4f}  accuracy: {accuracy:.3f}")

---

## 3. Refresh the iframe in-place

The dashboard auto-polls, but if you want to force an immediate reload without opening a new iframe, call `refresh()`.  
It updates the **same cell output** — no second iframe stacks up below.

In [ ]:
viewer.refresh()

---

## 4. Compare multiple experiments

Log several runs with different configs. The dashboard overlays their curves automatically.

In [ ]:
CONFIGS = [
    {"name": "sgd_lr0.1",   "lr": 0.10, "seed": 0},
    {"name": "sgd_lr0.01",  "lr": 0.01, "seed": 1},
    {"name": "sgd_lr0.001", "lr": 0.001,"seed": 2},
]
STEPS = 200

for cfg in CONFIGS:
    random.seed(cfg["seed"])
    lr = cfg["lr"]

    with SummaryWriter(f"runs/{cfg['name']}", config=cfg) as writer:
        for step in range(STEPS):
            decay = math.exp(-step * lr * 0.5)
            floor = 0.05 + lr * 0.3
            loss  = decay * 2.0 + floor + random.gauss(0, lr * 0.1)
            acc   = max(0.0, min(1.0, 1.0 - loss * 0.4 + random.gauss(0, 0.01)))
            writer.add_scalar("loss", loss, step)
            writer.add_scalar("accuracy", acc, step)

    print(f"  {cfg['name']:>15s}: final loss={loss:.4f}  acc={acc:.4f}")

print("\nAll runs logged. Refresh the dashboard to see them overlaid.")

In [ ]:
viewer.refresh()

---

## 5. Log hyperparameters

`add_hparams` attaches config + final metrics to each run. They appear in the **HParams** tab of the dashboard.

In [ ]:
final_results = {
    "sgd_lr0.1":   {"loss": 0.38, "accuracy": 0.847},
    "sgd_lr0.01":  {"loss": 0.21, "accuracy": 0.912},
    "sgd_lr0.001": {"loss": 0.55, "accuracy": 0.784},
}

for cfg in CONFIGS:
    metrics = final_results[cfg["name"]]
    with SummaryWriter(f"runs/{cfg['name']}", resume=True) as writer:
        writer.add_hparams(
            hparam_dict={"lr": cfg["lr"], "optimizer": "sgd", "steps": STEPS},
            metric_dict=metrics,
        )
    print(f"  {cfg['name']:>15s}: {metrics}")

print("\nHParams tab now shows a sortable comparison table.")

---

## 6. Filter to specific experiments

Pass `experiments=[...]` to focus the iframe on a subset of runs.

In [ ]:
# Show only the two best runs side-by-side
viewer.show(
    height=600,
    experiments=["sgd_lr0.01", "sgd_lr0.1"],
)

---

## 7. Get the server URL

The `.url` property gives you the raw address, useful for custom embedding or sharing the link with a colleague on the same machine.

In [ ]:
print("Dashboard URL:", viewer.url)

# Embed with custom HTML (e.g. inside an OutputWidget or custom layout)
from IPython.display import IFrame
IFrame(src=viewer.url, width="100%", height=500)

In [ ]:
# start_in_thread() starts the server without displaying anything —
# useful when you want to decide how/where to render yourself.
viewer2 = JupyterOutput()
viewer2.start_in_thread(port=0)   # port=0 → free port auto-assigned
print("Headless server:", viewer2.url)

---

## 8. JupyterHub / remote environments

On JupyterHub or hosted notebooks (Colab, Kaggle, SageMaker Studio), `127.0.0.1` may not be reachable through the proxy.  
Use `display_mode="link"` to get a clickable URL you can copy into the browser manually, or adjust with the proxy path your platform provides.

In [ ]:
viewer.show(display_mode="link")

In [ ]:
# On JupyterHub with jupyter-server-proxy installed:
#
#   proxy_base = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "") + "proxy/"
#   proxy_url  = f"{proxy_base}{port}/"
#
#   from IPython.display import IFrame
#   IFrame(src=proxy_url, width="100%", height=700)

import os

hub_prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX")
if hub_prefix and viewer.url:
    port = viewer.url.rsplit(":", 1)[-1]
    proxy_url = f"{hub_prefix}proxy/{port}/"
    from IPython.display import IFrame
    display(IFrame(src=proxy_url, width="100%", height=700))  # noqa: F821
else:
    print("Not running on JupyterHub — proxy URL not applicable.")
    print("Dashboard URL:", viewer.url)

---

## CLI equivalent

Everything above can also be launched from a terminal (or a notebook shell cell):

```bash
# Default — picks port 6116, binds to 127.0.0.1
vibetrack --viewer=jupyter

# Custom port + height (height is Jupyter-only; ignored in terminal fallback)
vibetrack --viewer=jupyter --port 7000

# Project-scoped
vibetrack runs/lr_search --viewer=jupyter
```

When called from a plain terminal (not inside an IPython kernel), `--viewer=jupyter` automatically falls back to the standard web viewer — identical to `--viewer=web`.